# Differential Evolution

For detailed description refer to [Wikipedia article](https://en.wikipedia.org/wiki/Differential_evolution).

In [1]:
# Import path to source directory (bit of a hack in Jupyter)
import sys
import os
pwd = %pwd
sys.path.append(os.path.join(pwd, '..', 'src'))

%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from objfun_dejong1 import DeJong1
from objfun_rastrigin import Rastrigin
from heur_de import DifferentialEvolution
from heur_fsa import FastSimulatedAnnealing
from heur_aux import CauchyMutation, MirrorCorrection

## Demo

In [3]:
dj = DeJong1(n=3, eps=0.01)
de = DifferentialEvolution(of=dj, maxeval=1000, N=10, CR=0.5, F=0.9)
res = de.search()
print('x_best = {}'.format(res['best_x']))
print('y_best = {}'.format(res['best_y']))
print('neval  = {}'.format(res['neval']))

x_best = [-0.06667781 -0.02799388  0.03464434]
y_best = 0.006429817846706898
neval  = 158


## Exercises

* Study the algorithm performance across mutation strategies
* Compare `rand/1`, `best/1`, and `current-to-best/1` (see e.g. [this SO question](http://stackoverflow.com/questions/20393102/all-versions-of-differential-evolution-algorithm) for reference)

## Tuning and evaluation of Differential Evolution

In [4]:
NUM_RUNS = 1000
MAXEVAL = 500


def experiment_de(of, maxeval, num_runs, N, CR, F, mutation, label):
    results = []
    for i in tqdm(range(num_runs), desc=f'Testing {label}'):
        result = DifferentialEvolution(of, maxeval=maxeval, N=N, CR=CR, F=F,
                                        mutation=mutation, seed=i).search()
        result['run'] = i
        result['heur'] = label
        results.append(result)
    return pd.DataFrame(results, columns=['heur', 'run', 'best_x', 'best_y', 'neval', 'success'])


def experiment_fsa(of, maxeval, num_runs, T0, n0, alpha, r, label):
    results = []
    for i in tqdm(range(num_runs), desc=f'Testing {label}'):
        result = FastSimulatedAnnealing(of, maxeval=maxeval, T0=T0, n0=n0, alpha=alpha,
                                        mutation=CauchyMutation(r=r, correction=MirrorCorrection(of)),
                                        seed=i).search()
        result['run'] = i
        result['heur'] = label
        results.append(result)
    return pd.DataFrame(results, columns=['heur', 'run', 'best_x', 'best_y', 'neval', 'success'])


def compute_stats(table, group_by='heur'):
    def agg(g):
        rel = g['success'].mean()
        successful = g.loc[g['success'], 'neval']
        mne = successful.mean() if len(successful) > 0 else np.nan
        feo = mne / rel if (rel > 0 and not np.isnan(mne)) else np.nan
        return pd.Series({'rel': rel, 'mne': mne, 'feo': feo})
    return table.groupby(group_by)[['neval', 'success']].apply(agg).reset_index()

### Population size N (DE/rand/1)

Fix `CR=0.9`, `F=0.8`. Include FSA as baseline.

In [5]:
results = pd.DataFrame()

for N in [4, 5, 7, 10, 15, 30]:
    res = experiment_de(dj, MAXEVAL, NUM_RUNS, N=N, CR=0.9, F=0.8, mutation='rand/1',
                        label=f'DE/rand/1 N={N} CR=0.9 F=0.8')
    res['N'] = N
    results = pd.concat([results, res], ignore_index=True)

for T0 in [1, 10, 100]:
    label = f'FSA T0={T0} n0=10 alpha=2 r=0.1'
    res = experiment_fsa(dj, MAXEVAL, NUM_RUNS, T0=T0, n0=10, alpha=2, r=0.1, label=label)
    results = pd.concat([results, res], ignore_index=True)

Testing DE/rand/1 N=4 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=5 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=7 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=10 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=15 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=30 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing FSA T0=1 n0=10 alpha=2 r=0.1:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing FSA T0=10 n0=10 alpha=2 r=0.1:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing FSA T0=100 n0=10 alpha=2 r=0.1:   0%|          | 0/1000 [00:00<?, ?it/s]

In [6]:
compute_stats(results).sort_values('feo')

,heur,rel,mne,feo
6,FSA T0=1 n0=10 alpha=2 r=0.1,1.000,118.495000,118.495000
7,FSA T0=10 n0=10 alpha=2 r=0.1,1.000,190.788000,190.788000
0,DE/rand/1 N=10 CR=0.9 F=0.8,0.996,245.158635,246.143207
5,DE/rand/1 N=7 CR=0.9 F=0.8,0.817,201.943696,247.177107
1,DE/rand/1 N=15 CR=0.9 F=0.8,0.981,350.747197,357.540466
3,DE/rand/1 N=4 CR=0.9 F=0.8,0.384,148.588542,386.949327
8,FSA T0=100 n0=10 alpha=2 r=0.1,0.851,329.957697,387.729373
4,DE/rand/1 N=5 CR=0.9 F=0.8,0.237,143.443038,605.244886
2,DE/rand/1 N=30 CR=0.9 F=0.8,0.084,422.797619,5033.304989


### Crossover rate CR (DE/rand/1)

Fix `N=10`, `F=0.8`.

In [7]:
for CR in [0.25, 0.50, 0.90]:
    res = experiment_de(dj, MAXEVAL, NUM_RUNS, N=10, CR=CR, F=0.8, mutation='rand/1',
                        label=f'DE/rand/1 N=10 CR={CR} F=0.8')
    res['CR'] = CR
    results = pd.concat([results, res], ignore_index=True)

compute_stats(results).sort_values('feo')

Testing DE/rand/1 N=10 CR=0.25 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=10 CR=0.5 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=10 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

,heur,rel,mne,feo
8,FSA T0=1 n0=10 alpha=2 r=0.1,1.000,118.495000,118.495000
9,FSA T0=10 n0=10 alpha=2 r=0.1,1.000,190.788000,190.788000
0,DE/rand/1 N=10 CR=0.25 F=0.8,1.000,243.292000,243.292000
1,DE/rand/1 N=10 CR=0.5 F=0.8,1.000,244.584000,244.584000
2,DE/rand/1 N=10 CR=0.9 F=0.8,0.996,245.158635,246.143207
7,DE/rand/1 N=7 CR=0.9 F=0.8,0.817,201.943696,247.177107
3,DE/rand/1 N=15 CR=0.9 F=0.8,0.981,350.747197,357.540466
5,DE/rand/1 N=4 CR=0.9 F=0.8,0.384,148.588542,386.949327
10,FSA T0=100 n0=10 alpha=2 r=0.1,0.851,329.957697,387.729373
6,DE/rand/1 N=5 CR=0.9 F=0.8,0.237,143.443038,605.244886


### Differential weight F (DE/rand/1)

Fix `N=10`, `CR=0.9`.

In [8]:
for F in [0.2, 0.4, 0.6, 0.8]:
    res = experiment_de(dj, MAXEVAL, NUM_RUNS, N=10, CR=0.9, F=F, mutation='rand/1',
                        label=f'DE/rand/1 N=10 CR=0.9 F={F}')
    res['F'] = F
    results = pd.concat([results, res], ignore_index=True)

compute_stats(results).sort_values('feo')

Testing DE/rand/1 N=10 CR=0.9 F=0.2:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=10 CR=0.9 F=0.4:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=10 CR=0.9 F=0.6:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/rand/1 N=10 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

,heur,rel,mne,feo
11,FSA T0=1 n0=10 alpha=2 r=0.1,1.000,118.495000,118.495000
12,FSA T0=10 n0=10 alpha=2 r=0.1,1.000,190.788000,190.788000
4,DE/rand/1 N=10 CR=0.9 F=0.6,0.877,188.930445,215.428101
0,DE/rand/1 N=10 CR=0.25 F=0.8,1.000,243.292000,243.292000
1,DE/rand/1 N=10 CR=0.5 F=0.8,1.000,244.584000,244.584000
5,DE/rand/1 N=10 CR=0.9 F=0.8,0.996,245.158635,246.143207
10,DE/rand/1 N=7 CR=0.9 F=0.8,0.817,201.943696,247.177107
6,DE/rand/1 N=15 CR=0.9 F=0.8,0.981,350.747197,357.540466
8,DE/rand/1 N=4 CR=0.9 F=0.8,0.384,148.588542,386.949327
13,FSA T0=100 n0=10 alpha=2 r=0.1,0.851,329.957697,387.729373


### DE/best/1 and the step-size problem

DE/best/1 uses the population best as base: `y = best + F*(b−c)`.

The step magnitude is `F × ‖b−c‖`. With a random initial population in [−5.12, 5.12]³,
the expected per-dimension spread `E[|b−c|] ≈ 3.4`, giving an initial step of `F×3.4≈2.7` per
dimension — **27× larger than FSA’s `r = 0.1`**. The step only shrinks once the
population converges, which is slow.

In [9]:
for N in [5, 10, 20, 30]:
    res = experiment_de(dj, MAXEVAL, NUM_RUNS, N=N, CR=0.9, F=0.8, mutation='best/1',
                        label=f'DE/best/1 N={N} CR=0.9 F=0.8')
    res['N'] = N
    results = pd.concat([results, res], ignore_index=True)

for F in [0.2, 0.4, 0.6, 0.8]:
    res = experiment_de(dj, MAXEVAL, NUM_RUNS, N=10, CR=0.9, F=F, mutation='best/1',
                        label=f'DE/best/1 N=10 CR=0.9 F={F}')
    res['F'] = F
    results = pd.concat([results, res], ignore_index=True)

compute_stats(results).sort_values('feo')

Testing DE/best/1 N=5 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/best/1 N=10 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/best/1 N=20 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/best/1 N=30 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/best/1 N=10 CR=0.9 F=0.2:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/best/1 N=10 CR=0.9 F=0.4:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/best/1 N=10 CR=0.9 F=0.6:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/best/1 N=10 CR=0.9 F=0.8:   0%|          | 0/1000 [00:00<?, ?it/s]

,heur,rel,mne,feo
18,FSA T0=1 n0=10 alpha=2 r=0.1,1.000,118.495000,118.495000
2,DE/best/1 N=10 CR=0.9 F=0.6,0.905,147.711602,163.217240
3,DE/best/1 N=10 CR=0.9 F=0.8,0.988,172.279352,174.371814
19,FSA T0=10 n0=10 alpha=2 r=0.1,1.000,190.788000,190.788000
11,DE/rand/1 N=10 CR=0.9 F=0.6,0.877,188.930445,215.428101
1,DE/best/1 N=10 CR=0.9 F=0.4,0.286,66.206294,231.490537
7,DE/rand/1 N=10 CR=0.25 F=0.8,1.000,243.292000,243.292000
8,DE/rand/1 N=10 CR=0.5 F=0.8,1.000,244.584000,244.584000
12,DE/rand/1 N=10 CR=0.9 F=0.8,0.996,245.158635,246.143207
17,DE/rand/1 N=7 CR=0.9 F=0.8,0.817,201.943696,247.177107


### DE/current-to-best/1 — fixing the step-size problem

Mutation: `y = x + F*(best − x) + F*(b − c)`

The `F*(best−x)` term provides a corrective pull **proportional to the distance from the best**
— large when far away, near-zero when close. For a sphere with best≈0:
`y ≈ (1−F)·x + noise` ⇒ `‖x‖` shrinks by factor `(1−F)` per generation.

**The role of CR.** CR controls a trade-off for this strategy:
* Higher CR → more dimensions get the convergence pull each trial (faster when it works)
* Higher CR → more dimensions also receive the `F*(b−c)` diversity noise each trial (more risk of overshooting → lower REL)

`CR=1.0` is therefore *not* the obvious winner: it amplifies both terms equally. The
experiments below show that **`CR≈0.5` gives the best balance**: most dimensions converge
reliably, while the noise term is limited to about half of them per trial.

In [10]:
# F sweep with CR=0.9 (reuse results already accumulated)
for F in [0.3, 0.5, 0.7, 0.9]:
    res = experiment_de(dj, MAXEVAL, NUM_RUNS, N=10, CR=0.9, F=F, mutation='current-to-best/1',
                        label=f'DE/c2b/1 N=10 CR=0.9 F={F}')
    res['F'] = F
    results = pd.concat([results, res], ignore_index=True)

# CR sweep for current-to-best/1 at the best F found above
for CR in [0.1, 0.5, 1.0]:
    res = experiment_de(dj, MAXEVAL, NUM_RUNS, N=10, CR=CR, F=0.5, mutation='current-to-best/1',
                        label=f'DE/c2b/1 N=10 CR={CR} F=0.5')
    res['CR'] = CR
    results = pd.concat([results, res], ignore_index=True)

compute_stats(results).sort_values('feo')

Testing DE/c2b/1 N=10 CR=0.9 F=0.3:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/c2b/1 N=10 CR=0.9 F=0.5:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/c2b/1 N=10 CR=0.9 F=0.7:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/c2b/1 N=10 CR=0.9 F=0.9:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/c2b/1 N=10 CR=0.1 F=0.5:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/c2b/1 N=10 CR=0.5 F=0.5:   0%|          | 0/1000 [00:00<?, ?it/s]

Testing DE/c2b/1 N=10 CR=1.0 F=0.5:   0%|          | 0/1000 [00:00<?, ?it/s]

,heur,rel,mne,feo
25,FSA T0=1 n0=10 alpha=2 r=0.1,1.000,118.495000,118.495000
10,DE/c2b/1 N=10 CR=0.9 F=0.5,0.823,106.166464,128.999349
13,DE/c2b/1 N=10 CR=1.0 F=0.5,0.704,96.691761,137.346252
11,DE/c2b/1 N=10 CR=0.9 F=0.7,0.945,130.776720,138.388063
8,DE/c2b/1 N=10 CR=0.5 F=0.5,0.986,136.997972,138.943176
9,DE/c2b/1 N=10 CR=0.9 F=0.3,0.835,127.035928,152.138836
2,DE/best/1 N=10 CR=0.9 F=0.6,0.905,147.711602,163.217240
3,DE/best/1 N=10 CR=0.9 F=0.8,0.988,172.279352,174.371814
7,DE/c2b/1 N=10 CR=0.1 F=0.5,1.000,177.292000,177.292000
12,DE/c2b/1 N=10 CR=0.9 F=0.9,0.999,184.000000,184.184184


### Conclusion — DeJong1

`DE/current-to-best/1` substantially closes the gap with FSA on this smooth unimodal function:

| Configuration | REL | FEO |
|---|---|---|
| FSA T0=1 (baseline) | 1.000 | **118** |
| DE/c2b/1 CR=0.5 F=0.5 | 0.986 | 139 |
| DE/c2b/1 CR=0.9 F=0.7 | 0.945 | 138 |
| DE/c2b/1 CR=0.1 F=0.5 | 1.000 | 177 |

The adaptive `F*(best−x)` pull is the key enabler. CR=0.5 strikes the best balance between
convergence speed and reliability. DE never quite matches FSA's FEO=118 with REL=1.0
simultaneously — the irreducible gap comes from the N=10 population overhead (10 evals per
round vs FSA's 1). But with `current-to-best/1` and tuned CR, DE is competitive (≈15–20%
worse FEO) rather than the ~50% gap seen with `rand/1` or `best/1`.

## Appendix 2 — Rastrigin n=2 (multimodal)

Compare with FSA from `04_FSA_dimensionality_scaling-Rastrigin.ipynb`:
same objective function (`n=2`, `eps=0.5`), same budget (`maxeval=4000`),
same FSA configurations (`FSA_r_scaled`, `FSA_tuned`).

The question: can DE maintain its population diversity advantage on a multimodal
landscape where FSA already achieves REL=1.0?

In [11]:
NUM_RUNS = 250
MAXEVAL = 4000

rastrigin2 = Rastrigin(n=2, eps=0.5)
results2 = pd.DataFrame()

# FSA baselines from notebook 04 (same parameters, same budget)
for label, T0, n0, r in [
    ('FSA_r_scaled T0=1 n0=100 r=0.354',  1, 100, 0.5 / np.sqrt(2)),
    ('FSA_tuned    T0=1 n0=200 r=0.354',  1, 200, 0.5 / np.sqrt(2)),
]:
    res = experiment_fsa(rastrigin2, MAXEVAL, NUM_RUNS, T0=T0, n0=n0, alpha=2, r=r, label=label)
    results2 = pd.concat([results2, res], ignore_index=True)

# DE sweeps
for mutation in ['rand/1', 'best/1', 'current-to-best/1']:
    for N in [10, 30]:
        label = f'DE/{mutation} N={N}'
        res = experiment_de(rastrigin2, MAXEVAL, NUM_RUNS, N=N, CR=0.9, F=0.8,
                            mutation=mutation, label=label)
        res['N'] = N
        results2 = pd.concat([results2, res], ignore_index=True)

Testing FSA_r_scaled T0=1 n0=100 r=0.354:   0%|          | 0/250 [00:00<?, ?it/s]

Testing FSA_tuned    T0=1 n0=200 r=0.354:   0%|          | 0/250 [00:00<?, ?it/s]

Testing DE/rand/1 N=10:   0%|          | 0/250 [00:00<?, ?it/s]

Testing DE/rand/1 N=30:   0%|          | 0/250 [00:00<?, ?it/s]

Testing DE/best/1 N=10:   0%|          | 0/250 [00:00<?, ?it/s]

Testing DE/best/1 N=30:   0%|          | 0/250 [00:00<?, ?it/s]

Testing DE/current-to-best/1 N=10:   0%|          | 0/250 [00:00<?, ?it/s]

Testing DE/current-to-best/1 N=30:   0%|          | 0/250 [00:00<?, ?it/s]

In [12]:
compute_stats(results2).sort_values('feo')

,heur,rel,mne,feo
4,DE/rand/1 N=10,0.592,366.006757,618.254657
1,DE/best/1 N=30,0.656,417.512195,636.451517
6,FSA_r_scaled T0=1 n0=100 r=0.354,1.000,823.392000,823.392000
2,DE/current-to-best/1 N=10,0.640,527.231250,823.798828
7,FSA_tuned T0=1 n0=200 r=0.354,1.000,834.728000,834.728000
0,DE/best/1 N=10,0.192,169.333333,881.944444
3,DE/current-to-best/1 N=30,0.996,891.265060,894.844438
5,DE/rand/1 N=30,1.000,965.208000,965.208000


### Conclusion — Rastrigin n=2

Comparing with FSA from notebook 04 (same function, budget, configurations):

| Configuration | REL | FEO |
|---|---|---|
| FSA_r_scaled (nb04 best) | 1.000 | **823** |
| FSA_tuned (nb04)         | 1.000 | 835 |
| DE/c2b/1 N=30            | 0.996 | 895 |
| DE/rand/1 N=30           | 1.000 | 965 |

FSA with a well-tuned mutation width (`r=0.354`, scaled from the unimodal result in notebook 03b)
still achieves the best FEO. However, DE is genuinely competitive here in a way it is not on
DeJong1 — the gap is ~9% (c2b/1) or ~17% (rand/1), compared to ~50% on the smooth function.

**Why the smaller gap on Rastrigin?** DE's population maintains diversity across multiple basins,
letting the algorithm collectively triangulate toward the global basin. FSA's single trajectory
must rely on temperature to escape traps, and its advantage (no population overhead) is partially
offset by the risk of getting trapped in a local minimum.

**Small-N caution.** Configurations like `DE/rand/1 N=10` (FEO=618) or `DE/best/1 N=30` (FEO=637)
look attractive in the sorted table but have REL<0.66 — the low FEO reflects only the successful
minority of runs, not overall reliability. On multimodal functions, REL is the primary concern.